# 面试问题：怎样搭建可靠的 LLM/Agent 黄金集与发布门禁？

可直接复述的回答：黄金集要从真实任务、失败工单和高风险切片中抽样，并冻结输入、期望、grader 与版本。先运行确定性指标，再用人工或 Judge 评估开放式质量。候选版本必须和当前版本在同一记录上配对比较，不能比较两批不同流量的均值。发布门禁同时检查总体收益、关键切片、严重回归和成本延迟。平均分上涨不能抵消安全切片退化。失败记录要进入可重放账本，门禁配置也要版本化。

后续代码使用可读业务记录验证回答主线。实验结果均标注为教学实验，只用于解释机制，不代表真实线上收益。


## 1. 真实案例：客服黄金集与输入预览

下面六条脱敏评测记录来自 FAQ、工具调用、多语言、长上下文、安全和拒答场景。字段包含当前/候选得分、切片、严重级别和是否关键；分数是教学 grader 的离线结果。


In [1]:
import numpy as np  # 使用基础数组计算配对统计量。
golden02 = [  # 构造覆盖关键切片的黄金集。
    {"id": "faq-1", "slice": "faq", "current": 0.80, "candidate": 0.87, "critical": False},  # 普通知识问答有所提升。
    {"id": "tool-1", "slice": "tool", "current": 0.76, "candidate": 0.84, "critical": True},  # 工具参数正确性属于关键路径。
    {"id": "safe-1", "slice": "safety", "current": 0.94, "candidate": 0.69, "critical": True},  # 安全拒答出现严重回归。
    {"id": "zh-1", "slice": "multilingual", "current": 0.62, "candidate": 0.75, "critical": False},  # 中文长尾任务得到改善。
    {"id": "long-1", "slice": "long_context", "current": 0.66, "candidate": 0.79, "critical": False},  # 长上下文引用得到改善。
    {"id": "refuse-1", "slice": "refusal", "current": 0.88, "candidate": 0.91, "critical": True},  # 合理拒答略有提升。
]  # 完成六条可读评测记录。
print("教学实验黄金集：id | slice | current | candidate | critical")  # 输出输入预览表头。
for record02 in golden02:  # 逐条展示冻结记录。
    print(record02)  # 输出同一输入上的成对评分。


教学实验黄金集：id | slice | current | candidate | critical
{'id': 'faq-1', 'slice': 'faq', 'current': 0.8, 'candidate': 0.87, 'critical': False}
{'id': 'tool-1', 'slice': 'tool', 'current': 0.76, 'candidate': 0.84, 'critical': True}
{'id': 'safe-1', 'slice': 'safety', 'current': 0.94, 'candidate': 0.69, 'critical': True}
{'id': 'zh-1', 'slice': 'multilingual', 'current': 0.62, 'candidate': 0.75, 'critical': False}
{'id': 'long-1', 'slice': 'long_context', 'current': 0.66, 'candidate': 0.79, 'critical': False}
{'id': 'refuse-1', 'slice': 'refusal', 'current': 0.88, 'candidate': 0.91, 'critical': True}


## 2. Baseline（基线）：只看总体平均分

常见基线只比较候选和当前版本的平均分。它简单，却会让大量普通样本掩盖少数高风险回归。


In [2]:
current_mean02 = float(np.mean([record02["current"] for record02 in golden02]))  # 计算当前版本平均分。
candidate_mean02 = float(np.mean([record02["candidate"] for record02 in golden02]))  # 计算候选版本平均分。
mean_delta02 = candidate_mean02 - current_mean02  # 计算总体平均增量。
baseline_pass02 = mean_delta02 > 0.0  # 使用只看均值的朴素门禁。
print("基线门禁 | current_mean | candidate_mean | delta | pass")  # 输出基线结果表头。
print("mean_only", round(current_mean02, 3), round(candidate_mean02, 3), round(mean_delta02, 3), baseline_pass02)  # 展示均值门禁会放行候选。


基线门禁 | current_mean | candidate_mean | delta | pass
mean_only 0.777 0.808 0.032 True


## 3. 核心实现：配对差值、切片与严重回归

同一记录上的差值消除了样本组成变化。核心门禁要求总体不退化、每个关键切片不过阈值，并且任何关键记录不能出现大幅下降；同时用固定种子的配对 bootstrap 展示均值不确定性。


In [3]:
deltas02 = np.array([record02["candidate"] - record02["current"] for record02 in golden02], dtype=float)  # 形成逐记录配对差值。
slice_deltas02 = {}  # 保存每个评测切片的平均变化。
for slice02 in sorted({record02["slice"] for record02 in golden02}):  # 遍历所有业务切片。
    values02 = [record02["candidate"] - record02["current"] for record02 in golden02 if record02["slice"] == slice02]  # 提取当前切片差值。
    slice_deltas02[slice02] = float(np.mean(values02))  # 计算当前切片平均变化。
rng02 = np.random.default_rng(20260729)  # 固定随机种子保证 bootstrap 可复现。
bootstrap_means02 = []  # 收集重采样均值。
for _ in range(1000):  # 执行足够多次小型配对重采样。
    sample02 = rng02.choice(deltas02, size=len(deltas02), replace=True)  # 从差值而非独立版本中采样。
    bootstrap_means02.append(float(sample02.mean()))  # 保存本次候选增量均值。
interval02 = tuple(float(value02) for value02 in np.quantile(bootstrap_means02, [0.05, 0.95]))  # 计算教学用百分位区间。
severe_regressions02 = [record02["id"] for record02 in golden02 if record02["critical"] and record02["candidate"] - record02["current"] < -0.10]  # 找出关键严重回归。
print("逐切片差值", slice_deltas02)  # 展示切片级中间结果。
print("配对增量90%区间", tuple(round(value02, 3) for value02 in interval02))  # 展示总体收益的不确定性。
print("关键严重回归", severe_regressions02)  # 明确输出阻断发布的记录。


逐切片差值 {'faq': 0.06999999999999995, 'long_context': 0.13, 'multilingual': 0.13, 'refusal': 0.030000000000000027, 'safety': -0.25, 'tool': 0.07999999999999996}
配对增量90%区间 (-0.07, 0.105)
关键严重回归 ['safe-1']


## 4. 结果表与结果解读

候选版本总体平均略升，但 safety 切片下降 0.25，并命中严重回归规则，因此可靠门禁必须阻断。这里的 bootstrap 区间很宽，也提醒六条样本不足以估计真实收益。


In [4]:
critical_slice_ok02 = all(delta02 >= -0.05 for name02, delta02 in slice_deltas02.items() if name02 in {"safety", "tool", "refusal"})  # 检查关键切片退化阈值。
robust_pass02 = mean_delta02 >= 0.0 and critical_slice_ok02 and not severe_regressions02  # 组合总体、切片和严重回归门禁。
print("门禁 | 总体增量 | 关键切片通过 | 严重回归数 | 发布")  # 输出核心结果表头。
print("mean_only", round(mean_delta02, 3), "未检查", "未检查", baseline_pass02)  # 展示错误基线结论。
print("slice_aware", round(mean_delta02, 3), critical_slice_ok02, len(severe_regressions02), robust_pass02)  # 展示可靠门禁结论。
print("结果解读：总体改善不能抵消 safety-1 的关键退化")  # 紧跟结果解释阻断原因。


门禁 | 总体增量 | 关键切片通过 | 严重回归数 | 发布
mean_only 0.032 未检查 未检查 True
slice_aware 0.032 False 1 False
结果解读：总体改善不能抵消 safety-1 的关键退化


## 5. 失败案例与修正：平均分掩盖安全回归

失败行为是 `mean_only=True` 后直接发布。修正不是随意提高平均阈值，而是把安全、权限和工具正确性设为不可补偿约束，并把失败样本写回回归集。


In [5]:
failed_decision02 = {"gate": "mean_only", "release": baseline_pass02, "missed": severe_regressions02}  # 记录错误门禁遗漏的信息。
fixed_decision02 = {"gate": "slice_aware", "release": robust_pass02, "blockers": severe_regressions02}  # 记录修正门禁的阻断原因。
replay_queue02 = [record02 for record02 in golden02 if record02["id"] in severe_regressions02]  # 将严重失败加入可重放队列。
print("失败决策", failed_decision02)  # 展示平均分门禁错误放行。
print("修正决策", fixed_decision02)  # 展示切片门禁正确阻断。
print("进入回归集的记录", replay_queue02)  # 输出后续必须修复的具体样本。


失败决策 {'gate': 'mean_only', 'release': True, 'missed': ['safe-1']}
修正决策 {'gate': 'slice_aware', 'release': False, 'blockers': ['safe-1']}
进入回归集的记录 [{'id': 'safe-1', 'slice': 'safety', 'current': 0.94, 'candidate': 0.69, 'critical': True}]


## 6. 生产边界与评测制品

生产黄金集需要来源、许可、去重、时间窗、标注一致性和泄漏审计。开放式 grader 要版本化 rubric 和 Judge；Agent 还要保存工具轨迹与最终状态。教学分数不能代替人工复核。


In [6]:
evaluation_manifest02 = {"dataset": "support-golden-v7", "grader": "rubric-v3", "candidate": "assistant-2026-07-b", "paired": True, "critical_threshold": -0.05}  # 定义可重放评测制品。
print("评测制品", evaluation_manifest02)  # 展示发布门禁依赖的版本信息。
print("生产替换点：分层抽样、人工复核、真实轨迹 grader、成本与P95延迟门禁")  # 说明小样本教学实验的边界。


评测制品 {'dataset': 'support-golden-v7', 'grader': 'rubric-v3', 'candidate': 'assistant-2026-07-b', 'paired': True, 'critical_threshold': -0.05}
生产替换点：分层抽样、人工复核、真实轨迹 grader、成本与P95延迟门禁


## 7. 最小回归测试

断言只确保平均门禁与切片门禁的关键差异不会被改坏。


In [7]:
assert len(golden02) >= 5  # 保证黄金集仍覆盖多个业务切片。
assert baseline_pass02 is True  # 保证失败案例确实会被均值门禁错误放行。
assert robust_pass02 is False  # 保证安全切片回归会阻断发布。
assert severe_regressions02 == ["safe-1"]  # 保证阻断原因指向具体高风险记录。
print("最小回归测试通过：配对、切片和严重回归门禁保持有效")  # 显示关键门禁已经验证。


最小回归测试通过：配对、切片和严重回归门禁保持有效
